# ws-gamefinaltest Seed Data

Run this notebook in Fabric after the local security-group step has produced `teams-resolved.xlsx`. The notebook only needs Fabric REST access because the workbook already contains `SecurityGroupId` values.


In [ ]:
# Configuration — edit these
EXCEL_PATH = "Files/teams-resolved.xlsx"
WORKSPACE_PREFIX = "bis-day"
CAPACITY_ID = "21F29096-B423-437C-AB24-60F1830444A2"
DRY_RUN = True

In [ ]:
import pandas as pd

df = pd.read_excel(f"/lakehouse/default/{EXCEL_PATH}")
required_columns = {"TeamName", "SecurityGroupId", "SecurityGroupName"}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(
        "The workbook must include SecurityGroupName and SecurityGroupId columns. "
        "Run 01-create-security-groups.py locally first."
    )

team_rows = df.drop_duplicates(subset=["TeamName"])[["TeamName", "SecurityGroupId", "SecurityGroupName"]]
team_rows = team_rows.to_dict(orient="records")
print(f"Found {len(team_rows)} teams:")
for row in team_rows:
    print(f"  {row['TeamName']}: {row['SecurityGroupName']} ({row['SecurityGroupId']})")

In [ ]:
import notebookutils

fabric_token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
fabric_headers = {
    "Authorization": f"Bearer {fabric_token}",
    "Content-Type": "application/json",
}

In [ ]:
import requests
import time

results = []

def find_workspace_id_by_name(display_name: str) -> str | None:
    # Handle paged workspace listings so reruns can reliably resolve existing IDs.
    url = "https://api.fabric.microsoft.com/v1/workspaces"
    while url:
        resp = requests.get(url, headers=fabric_headers, timeout=60)
        if resp.status_code != 200:
            return None
        payload = resp.json()
        value = payload.get("value", [])
        for workspace in value:
            if workspace.get("displayName") == display_name:
                return workspace.get("id")
        next_link = payload.get("@odata.nextLink") or payload.get("nextLink")
        url = next_link if isinstance(next_link, str) and next_link else None
    return None

for row in team_rows:
    team_name = row["TeamName"]
    ws_name = f"{WORKSPACE_PREFIX}-{team_name}"
    sg_name = row["SecurityGroupName"]
    sg_id = row["SecurityGroupId"]

    print(f"\n{'='*50}")
    print(f"Team: {team_name}")
    print(f"  Workspace: {ws_name}")
    print(f"  Security Group: {sg_name}")

    if DRY_RUN:
        print(f"  [DRY RUN] Would create workspace '{ws_name}', assign capacity, add '{sg_name}' as Contributor")
        results.append({"team": team_name, "workspace": ws_name, "status": "dry_run"})
        continue

    ws_resp = requests.post(
        "https://api.fabric.microsoft.com/v1/workspaces",
        headers=fabric_headers,
        json={"displayName": ws_name},
        timeout=60,
    )

    if ws_resp.status_code == 201:
        ws_id = ws_resp.json()["id"]
        print(f"  ✅ Created workspace: {ws_name} (ID: {ws_id})")
    elif ws_resp.status_code == 409:
        ws_id = find_workspace_id_by_name(ws_name)
        if ws_id:
            print(f"  ⚠️ Workspace already exists: {ws_name} (ID: {ws_id})")
        else:
            print("  ❌ Workspace conflict but couldn't find it")
            results.append({"team": team_name, "workspace": ws_name, "status": "conflict_error"})
            continue
    else:
        print(f"  ❌ Failed to create workspace: {ws_resp.status_code} {ws_resp.text}")
        results.append({"team": team_name, "workspace": ws_name, "status": "failed", "error": ws_resp.text})
        continue

    cap_resp = requests.post(
        f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/assignToCapacity",
        headers=fabric_headers,
        json={"capacityId": CAPACITY_ID},
        timeout=60,
    )
    if cap_resp.status_code in (200, 202):
        print(f"  ✅ Assigned to capacity: {CAPACITY_ID}")
    else:
        print(f"  ⚠️ Capacity assignment: {cap_resp.status_code} {cap_resp.text}")

    role_body = {
        "principal": {"id": sg_id, "type": "Group"},
        "role": "Contributor",
    }
    role_resp = requests.post(
        f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/roleAssignments",
        headers=fabric_headers,
        json=role_body,
        timeout=60,
    )

    if role_resp.status_code in (200, 201):
        print(f"  ✅ Added '{sg_name}' as Contributor")
        results.append({"team": team_name, "workspace": ws_name, "ws_id": ws_id, "sg_id": sg_id, "status": "complete"})
    elif role_resp.status_code == 409:
        print("  ⚠️ Role already assigned")
        results.append({"team": team_name, "workspace": ws_name, "ws_id": ws_id, "sg_id": sg_id, "status": "complete_existing"})
    else:
        print(f"  ❌ Role assignment failed: {role_resp.status_code} {role_resp.text}")
        results.append({"team": team_name, "workspace": ws_name, "ws_id": ws_id, "status": "role_failed", "error": role_resp.text})

    time.sleep(0.5)

In [ ]:
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

if DRY_RUN:
    print("\n⚠️ DRY RUN — nothing was created. Set DRY_RUN = False and fill in CAPACITY_ID to execute.")
else:
    print(f"\n✅ Done! {len([r for r in results if 'complete' in r.get('status','')])} workspaces fully configured.")